# ClimateVision Regional Bias Audit

This notebook demonstrates how to evaluate model fairness across geographic regions.
Ensuring equitable predictions is critical for NGOs operating in different parts of the world.

**Author:** Linda Oraegbunam (@obielin)  
**Module:** `src/climatevision/governance/bias_audit.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from climatevision.governance import (
    run_bias_audit,
    BiasAuditor,
    BiasReport,
    check_fairness_gate,
    SUPPORTED_REGIONS,
)

## 1. Understanding Regional Bias

Climate models trained primarily on Amazon data may underperform on Congo Basin imagery due to:
- Different forest types and canopy structures
- Varying cloud patterns and seasonal effects
- Different satellite viewing angles and atmospheric conditions

This audit ensures NGOs in all regions receive equally reliable predictions.

In [ ]:
# View supported regions
print("Supported Regions for Bias Audit:")
print("=" * 50)
for key, info in SUPPORTED_REGIONS.items():
    print(f"\n{info['name']} ({key})")
    print(f"  Bounding Box: {info['bbox']}")
    print(f"  Description: {info['description']}")

## 2. Creating a Bias Auditor

In [ ]:
# Create auditor with 85% fairness threshold
auditor = BiasAuditor(model=None, threshold=0.85)

# Simulate regional prediction data
# In production, this would be real model outputs on test sets
np.random.seed(42)

regions_data = {
    'amazon': {'accuracy': 0.92, 'forest_ratio': 0.70},
    'congo': {'accuracy': 0.85, 'forest_ratio': 0.65},
    'southeast_asia': {'accuracy': 0.88, 'forest_ratio': 0.55},
}

for region, params in regions_data.items():
    n_samples = 1000
    
    # Ground truth based on regional forest coverage
    ground_truth = (np.random.random(n_samples) < params['forest_ratio']).astype(int)
    
    # Predictions based on regional accuracy
    correct = np.random.random(n_samples) < params['accuracy']
    predictions = np.where(correct, ground_truth, 1 - ground_truth)
    
    auditor.add_region_data(region, predictions, ground_truth)
    print(f"Added {n_samples} samples for {region}")

## 3. Computing Fairness Metrics

In [ ]:
# Run full bias audit
report = auditor.run_audit(
    metric='equalized_odds',
    model_path='models/demo_model.pth',
    model_version='v1.0-demo',
    analysis_type='deforestation',
)

print(f"Fairness Score: {report.fairness_score:.4f}")
print(f"Threshold: {report.threshold}")
print(f"Passed: {'✅' if report.passed else '❌'}")
print(f"\nDisparity Regions: {report.disparity_regions or 'None'}")

In [ ]:
# View per-region metrics
print("Per-Region Metrics:")
print("=" * 60)

for metrics in report.region_metrics:
    print(f"\n{metrics.region_name} ({metrics.region}):")
    print(f"  Samples: {metrics.n_samples}")
    print(f"  IoU: {metrics.iou:.4f}")
    print(f"  F1: {metrics.f1:.4f}")
    print(f"  Precision: {metrics.precision:.4f}")
    print(f"  Recall: {metrics.recall:.4f}")
    print(f"  TPR: {metrics.true_positive_rate:.4f}")
    print(f"  FPR: {metrics.false_positive_rate:.4f}")

## 4. Visualizing Regional Disparities

In [ ]:
# Prepare data for visualization
regions = [m.region_name for m in report.region_metrics]
ious = [m.iou for m in report.region_metrics]
f1s = [m.f1 for m in report.region_metrics]
tprs = [m.true_positive_rate for m in report.region_metrics]

x = np.arange(len(regions))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width, ious, width, label='IoU', color='#3498db')
bars2 = ax.bar(x, f1s, width, label='F1 Score', color='#2ecc71')
bars3 = ax.bar(x + width, tprs, width, label='True Positive Rate', color='#e74c3c')

ax.set_ylabel('Score')
ax.set_title('Model Performance by Region')
ax.set_xticks(x)
ax.set_xticklabels(regions)
ax.legend()
ax.set_ylim(0, 1.1)
ax.axhline(y=0.85, color='gray', linestyle='--', label='Threshold')

plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for multi-metric comparison
from math import pi

categories = ['IoU', 'F1', 'Precision', 'Recall', 'TPR']
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, metrics in enumerate(report.region_metrics):
    values = [metrics.iou, metrics.f1, metrics.precision, metrics.recall, metrics.true_positive_rate]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=metrics.region_name, color=colors[i % len(colors)])
    ax.fill(angles, values, alpha=0.25, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.set_title('Regional Performance Comparison', y=1.08)

plt.tight_layout()
plt.show()

## 5. Comparing Fairness Metrics

In [ ]:
# Compare different fairness metrics
metrics_to_test = ['demographic_parity', 'equalized_odds', 'predictive_parity']
results = {}

for metric in metrics_to_test:
    report = auditor.run_audit(metric=metric)
    results[metric] = {
        'score': report.fairness_score,
        'passed': report.passed,
        'disparity_regions': report.disparity_regions,
    }

print("Fairness Metrics Comparison:")
print("=" * 50)
for metric, result in results.items():
    status = '✅' if result['passed'] else '❌'
    print(f"\n{metric}:")
    print(f"  Score: {result['score']:.4f} {status}")
    if result['disparity_regions']:
        print(f"  Disparity in: {', '.join(result['disparity_regions'])}")

## 6. Using the High-Level API

In [ ]:
# For real usage with trained models:
# result = run_bias_audit(
#     model_path='models/unet_deforestation.pth',
#     regions=['amazon', 'congo', 'southeast_asia'],
#     metric='equalized_odds',
#     threshold=0.85,
# )
# 
# print(f"Score: {result['score']}")
# print(f"Passed: {result['passed']}")
# print(f"Report: {result['report_path']}")

print("See run_bias_audit() for production usage")

## 7. CI/CD Integration

In [ ]:
# CI gate function for automated checks
# This would be called in GitHub Actions or similar

# passed = check_fairness_gate(
#     model_path='models/best_model.pth',
#     regions=['amazon', 'congo', 'southeast_asia'],
#     threshold=0.85,
# )
# 
# if not passed:
#     sys.exit(1)  # Fail the CI build

print("Use check_fairness_gate() in CI/CD pipelines")
print("Command: python scripts/audit_model.py --model models/best.pth --ci-gate")

## 8. Recommendations

In [ ]:
# Get recommendations from the audit
print("Recommendations:")
print("=" * 50)
for rec in report.recommendations:
    print(f"\n• {rec}")

## Summary

This notebook demonstrated:

1. **BiasAuditor** - Core class for fairness evaluation
2. **Fairness Metrics** - Demographic parity, equalized odds, predictive parity
3. **Regional Analysis** - Per-region IoU, F1, precision, recall
4. **Visualization** - Bar charts and radar plots for stakeholder reports
5. **CI/CD Integration** - `check_fairness_gate()` for automated checks

For production use:
- Run `python scripts/audit_model.py --model <path> --regions amazon,congo`
- Add `--ci-gate` flag to fail builds with poor fairness scores